In [14]:
import pandas as pd
import json
import datasets
import csv

In [6]:
json.__dict__

{'__name__': 'json',
 '__doc__': 'JSON (JavaScript Object Notation) <https://json.org> is a subset of\nJavaScript syntax (ECMA-262 3rd edition) used as a lightweight data\ninterchange format.\n\n:mod:`json` exposes an API familiar to users of the standard library\n:mod:`marshal` and :mod:`pickle` modules.  It is derived from a\nversion of the externally maintained simplejson library.\n\nEncoding basic Python object hierarchies::\n\n    >>> import json\n    >>> json.dumps([\'foo\', {\'bar\': (\'baz\', None, 1.0, 2)}])\n    \'["foo", {"bar": ["baz", null, 1.0, 2]}]\'\n    >>> print(json.dumps("\\"foo\\bar"))\n    "\\"foo\\bar"\n    >>> print(json.dumps(\'\\u1234\'))\n    "\\u1234"\n    >>> print(json.dumps(\'\\\\\'))\n    "\\\\"\n    >>> print(json.dumps({"c": 0, "b": 0, "a": 0}, sort_keys=True))\n    {"a": 0, "b": 0, "c": 0}\n    >>> from io import StringIO\n    >>> io = StringIO()\n    >>> json.dump([\'streaming API\'], io)\n    >>> io.getvalue()\n    \'["streaming API"]\'\n\nCompact e

In [ ]:
with open("feverdataset_train.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]


AttributeError: 'str' object has no attribute 'read'

In [9]:

df = pd.read_json("feverdataset_train.jsonl", lines=True)
df.head()

,id,verifiable,label,claim,evidence
0,75397,VERIFIABLE,SUPPORTS,Nikolaj Coster-Waldau worked with the Fox Broa...,"[[[92206, 104971, Nikolaj_Coster-Waldau, 7], [..."
1,150448,VERIFIABLE,SUPPORTS,Roman Atwood is a content creator.,"[[[174271, 187498, Roman_Atwood, 1]], [[174271..."
2,214861,VERIFIABLE,SUPPORTS,"History of art includes architecture, dance, s...","[[[255136, 254645, History_of_art, 2]]]"
3,156709,VERIFIABLE,REFUTES,Adrienne Bailon is an accountant.,"[[[180804, 193183, Adrienne_Bailon, 0]]]"
4,83235,NOT VERIFIABLE,NOT ENOUGH INFO,System of a Down briefly disbanded in limbo.,"[[[100277, None, None, None]]]"


In [10]:
refutes=df[df["label"]=="REFUTES"]
refutes['claim'].head(20)

3                     Adrienne Bailon is an accountant.
14      Stranger Things is set in Bloomington, Indiana.
16    Puerto Rico is not an unincorporated territory...
26    Peggy Sue Got Married is a Egyptian film relea...
27    Andy Roddick lost 5 Master Series between 2002...
28    As the Vietnam War raged in 1969, Yoko Ono and...
40                             Grace Jones is a dancer.
42    Willie Nelson dropped out of college after thr...
46    Furia is adapted from a short story by Anna Po...
51                C. S. Forester's first name was Carl.
52                        Kong: Skull Island is a book.
53                   The Challenge was a scripted show.
62                      Red Headed Stranger is a movie.
70                  Chester Bennington is not a singer.
73                 David Beckham has zero middle names.
74    Indiana Jones has only been portrayed by Harri...
78       Shane Black and Chris Miller wrote Iron Man 3.
79      La La Anthony was conceived on June 25th

In [11]:
len(refutes)

29775

In [12]:
refutes.to_csv("fever_refuted.csv")

In [ ]:
refute1="World War II did not have campaigns occurring in Asia"
refute2="Large numbers of civilians were not executed during the French Revolution."

In [15]:
_DESCRIPTION = """\
We present a human-and-model-in-the-loop process for dynamically generating datasets and training better performing and more robust hate detection models. We provide a new dataset of ~40,000 entries, generated and labelled by trained annotators over four rounds of dynamic data creation. It includes ~15,000 challenging perturbations and each hateful entry has fine-grained labels for the type and target of hate. Hateful entries make up 54% of the dataset, which is substantially higher than comparable datasets. We show that model performance is substantially improved using this approach. Models trained on later rounds of data collection perform better on test sets and are harder for annotators to trick. They also perform better on HATECHECK, a suite of functional tests for online hate detection. See https://arxiv.org/abs/2012.15761 for more details.
"""

_CITATION = """\
@inproceedings{vidgen2021learning,
  title={Learning from the Worst: Dynamically Generated Datasets to Improve Online Hate Detection},
  author={Vidgen, Bertie and Thrush, Tristan and Waseem, Zeerak and Kiela, Douwe},
  booktitle={Proceedings of the 59th Annual Meeting of the Association for Computational Linguistics and the 11th International Joint Conference on Natural Language Processing (Volume 1: Long Papers)},
  pages={1667--1682},
  year={2021}
}
"""

_DOWNLOAD_URL = "https://raw.githubusercontent.com/bvidgen/Dynamically-Generated-Hate-Speech-Dataset/main/Dynamically%20Generated%20Hate%20Dataset%20v{version}.csv"
_VERSIONS = ("0.2.3", "0.2.2")


class Dynahate(datasets.GeneratorBasedBuilder):
    """Learning from the worst hate speech classification dataset."""
    BUILDER_CONFIGS = [datasets.BuilderConfig(name=version, version=version) for version in _VERSIONS]
    DEFAULT_CONFIG_NAME = "0.2.3"
    def _info(self):
        return datasets.DatasetInfo(
            description=_DESCRIPTION,
            features=datasets.Features(
                {
                    "acl.id": datasets.Value("string"),
                    "label": datasets.features.ClassLabel(names=["nothate", "hate"]),
                    "text": datasets.Value("string"),
                    "X1": datasets.Value("int32"),
                    "type": datasets.Value("string"),
                    "target": datasets.Value("string"),
                    "level": datasets.Value("string"),
                    "split": datasets.Value("string"),
                    "round.base": datasets.Value("int32"),
                    "annotator": datasets.Value("string"),
                    "round": datasets.Value("string"),
                    "acl.id.matched": datasets.Value("string")
                }
            ),
            homepage="https://arxiv.org/abs/2012.15761",
            citation=_CITATION,
        )

    def _split_generators(self, dl_manager):
        csv_path = dl_manager.download_and_extract(_DOWNLOAD_URL.format(version=self.config.name))
        return [
            datasets.SplitGenerator(name=datasets.Split.TRAIN, gen_kwargs={"csv_path": csv_path, "split": "train"}),
            datasets.SplitGenerator(name=datasets.Split.VALIDATION, gen_kwargs={"csv_path": csv_path, "split": "dev"}),
            datasets.SplitGenerator(name=datasets.Split.TEST, gen_kwargs={"csv_path": csv_path, "split": "test"}),
        ]

    def _generate_examples(self, csv_path, split):
        """Generate AG News examples."""
        with open(csv_path, encoding="utf-8") as csv_file:
            csv_reader = csv.reader(csv_file, delimiter=",")
            for id_, row in enumerate(csv_reader):
                if id_ == 0:
                    keys = row[:]
                else:
                    res = dict([(k, v) for k, v in zip(keys, row) if k != ""])
                
                    for k in ["X1", "round.base"]:
                        res[k] = int(res[k])
                    yield id_ - 1, res



In [1]:

# Option 2: pandas directly (simpler)
import pandas as pd
url = "https://raw.githubusercontent.com/bvidgen/Dynamically-Generated-Hate-Speech-Dataset/main/Dynamically%20Generated%20Hate%20Dataset%20v0.2.3.csv"
df = pd.read_csv(url)
print(df.head())
print(df.columns)

   Unnamed: 0    acl.id  X1  \
0           1      acl1   1   
1           2     acl10   2   
2           3    acl100   3   
3           4   acl1000   4   
4           5  acl10000   5   

                                                text label      type  \
0                                dalits are lowlives  hate  notgiven   
1             gay people are a burden to our society  hate  notgiven   
2                              Arabs are not welcome  hate  notgiven   
3  I'm not saying we should actually eliminate he...  hate  notgiven   
4                       bananas are for black people  hate  notgiven   

     target     level  split  round.base   annotator round acl.id.matched  
0  notgiven  original  train           1  GNZuCtwed3     1            NaN  
1  notgiven  original  train           1  GNZuCtwed3     1            NaN  
2  notgiven  original  train           1  vDe7GN0NrL     1            NaN  
3  notgiven  original  train           1  oemYWm1Tjg     1            NaN  


In [4]:
hate=df[df['label']=='hate']
hate['text'].sample(20)

31917              She's got a black baby, it's disgusting
35505    I just saw this queer in town and he tried to ...
15224    For the last two months, we've been following ...
31675    This is exactly what the feminists are shoutin...
37826                 queers shouldn't be allowed to adopt
40174        I hope we get rid of all black people by 2030
15021                                    women are a virus
37143    if you are a man and you like men, you are not...
13660    U ain't white. Accept whatever injun or whatev...
18884    lets ask this question why Trump suck evil jew...
8400     I though paedo was spelt like poof and now I k...
30227    There were Jews who worked in the slave trade,...
30605    Frank Karmey was an astronomer who was fired f...
39852    I disagree with the rights of black people in ...
35715                                all gays are perverts
7951     WATever fucking noisy black chicks down my str...
2641     If we were to kill all the black women the wor.

In [13]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Download latest version
path = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")

print("Path to dataset files:", path)

100%|██████████| 41.0M/41.0M [00:02<00:00, 16.9MB/s]

Extracting files...


Path to dataset files: C:\Users\eglha\.cache\kagglehub\datasets\emineyetm\fake-news-detection-datasets\versions\1


In [14]:
fake_news_pth=r"C:\Users\eglha\OneDrive\Documents\lesson_plan_LLM\data\News _dataset\Fake.csv"
fake_news=pd.read_csv(fake_news_pth)
fake_news.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [25]:
fake_news['war'].sample(20)

KeyError: 'war'

In [26]:
def search_term(search_term):
        term = str(search_term).lower()
        for text in fake_news['text'].astype(str):
            if term in text.lower():
                return text
        return "Not found"

print(search_term("war"))


On Friday, it was revealed that former Milwaukee Sheriff David Clarke, who was being considered for Homeland Security Secretary in Donald Trump s administration, has an email scandal of his own.In January, there was a brief run-in on a plane between Clarke and fellow passenger Dan Black, who he later had detained by the police for no reason whatsoever, except that maybe his feelings were hurt. Clarke messaged the police to stop Black after he deplaned, and now, a search warrant has been executed by the FBI to see the exchanges.Clarke is calling it fake news even though copies of the search warrant are on the Internet. I am UNINTIMIDATED by lib media attempts to smear and discredit me with their FAKE NEWS reports designed to silence me,  the former sheriff tweeted.  I will continue to poke them in the eye with a sharp stick and bitch slap these scum bags til they get it. I have been attacked by better people than them #MAGA I am UNINTIMIDATED by lib media attempts to smear and discredit

In [32]:
def search_term(search_term):
    term = str(search_term).lower()
    results = []
    for text in fake_news['text'].astype(str):
        if term in text.lower():
            results.append(text)
    print(f"Total Items containing the search term:{search_term} are: {len(results)}")
    return "\n".join(results) if results else "Not found"

print(search_term("sandy hook"))

Total Items containing the search term:sandy hook are: 76
It almost seems like Donald Trump is trolling America at this point. In the beginning, when he tried to gaslight the country by insisting that the crowd at his inauguration was the biggest ever   or that it was even close to the last couple of inaugurations   we all kind of scratched our heads and wondered what kind of bullshit he was playing at. Then when he started appointing people to positions they had no business being in, we started to worry that this was going to go much worse than we had expected.After 11 months of Donald Trump pulling the rhetorical equivalent of whipping his dick out and slapping it on every table he gets near, I think it s time we address what s happening: Dude is a straight-up troll. He gets pleasure out of making other people uncomfortable or even seeing them in distress. He actively thinks up ways to piss off people he doesn t like.Let s set aside just for a moment the fact that that s the least pr